In [1]:

from pyspark.sql import functions as F

# Read directly from Bronze Lakehouse
air_df = spark.read.table(
    "NYC_Mobility_Analytics.NYC_Bronze_LH.dbo.openaq_pm25_daily"
)

print("Original records:", air_df.count())

# Clean and standardize data
clean_df = (
    air_df
    .withColumn("date", F.to_date("date"))
    .withColumn("location_id", F.col("location_id").cast("long"))
    .withColumn("sensor_id", F.col("sensor_id").cast("long"))
    .withColumn("value", F.col("value").cast("double"))
    .withColumn("coverage_percent", F.col("coverage_percent").cast("double"))
    .withColumn("parameter", F.lower(F.trim(F.col("parameter"))))
    .withColumn("station", F.trim(F.col("station")))
    .withColumn("unit", F.trim(F.col("unit")))
    .filter(F.col("date").isNotNull())
    .filter(F.col("value").isNotNull())
    .filter(F.col("value") >= 0)
    .filter(F.col("sensor_id").isNotNull())
    .filter(F.col("parameter") == "pm25")
    .dropDuplicates(["date", "sensor_id"])
)

# Save explicitly to Silver Lakehouse
(
    clean_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "NYC_Mobility_Analytics.NYC_Silver_LH.dbo.openaq_pm25_daily_clean"
    )
)

print("OpenAQ Silver data saved successfully!")
print("Clean records:", clean_df.count())

display(clean_df.orderBy("date"))


StatementMeta(, 284f02db-21fc-42b0-a5c1-4777b314b33e, 3, Finished, Available, Finished, False)

Original records: 31
OpenAQ Silver data saved successfully!
Clean records: 31


SynapseWidget(Synapse.DataFrame, caf14bb3-ea2c-4002-96f9-03b7c080b594)